# 05 - Validation against openrouteservice
Recompute the network metrics of notebook 04 for 5 random buildings using the openrouteservice (HeiGIT) API
(`foot-walking` matrix and isochrones) and compare them with the values in `output/buildings_network.gpkg`.
Nothing in the original process is changed; this notebook only reports the differences.

In [1]:
import os
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import requests

## Load data and sample 5 buildings
Same layers and filters as notebook 04. `fid` is the GeoPackage feature id (the `fid` field in QGIS).

In [2]:
CRS = 32721
WALK = 1200
gp = "data/geopackage"
ORS = "https://api.openrouteservice.org/v2"
HEAD = {"Authorization": os.environ["OPEN_ROUTE_SERVICE"]}

bld = gpd.read_file("output/buildings_network.gpkg", fid_as_index=True).to_crs(CRS)
sample = bld.sample(5, random_state=42)
sample.index.name = "fid"

read = lambda f: gpd.read_file(f"{gp}/{f}.gpkg").to_crs(CRS).reset_index(drop=True)
amenities, coast, green, transport = read("amenities"), read("coast_beaches"), read("green_areas"), read("public_transport")
coastline = coast[coast["natural"] == "coastline"]
beaches = coast[coast["natural"] == "beach"]
is_park = (green["leisure"].isin(["park", "garden", "playground"]) | (green["place"] == "square")).values
parks = green[is_park]
hubs = transport[(transport["amenity"] == "bus_station") | (transport["public_transport"] == "station")]
stops = transport[transport["highway"] == "bus_stop"]

sample[["s2_id", "geohash", "area_in_meters", "livability"]]

,s2_id,geohash,area_in_meters,livability
fid,,,,
5717,-7782220156096217088,6cbcq9ng,96.5040,36.0
34186,-7782220156096217088,6cbcmy0x,63.8458,45.1
42918,-7782220156096217088,6cbcmm9h,140.5635,71.7
31480,-7782220156096217088,6cbcmzjc,32.6933,34.0
75274,-7782220156096217088,6cbcm6vw,55.4486,72.0


## ORS helpers
ORS routes between points, so every feature is reduced to its point closest (straight line) to the building
(the building itself if it lies inside a polygon). Beaches and coastline are long features, so their outlines are
densified every 100 m and each vertex is a candidate. Only candidates within 1200 m straight line, plus the 49 nearest,
are sent to the matrix (in chunks of 49, respecting the 40 requests/min limit).

In [3]:
def matrix(p, targets):
    ll = gpd.GeoSeries([p, *targets], crs=CRS).to_crs(4326)
    locs = np.c_[ll.x, ll.y].tolist()
    out = []
    for i in range(1, len(locs), 49):
        chunk = locs[i:i + 49]
        r = requests.post(f"{ORS}/matrix/foot-walking", headers=HEAD, json={
            "locations": [locs[0]] + chunk, "sources": [0],
            "destinations": list(range(1, len(chunk) + 1)), "metrics": ["distance"]})
        r.raise_for_status()
        j = r.json()
        out += j["distances"][0]
        time.sleep(1.6)
    return np.array(out, dtype=float), j["sources"][0].get("snapped_distance", 0.0)

def ors_layer(p, geoms):
    t = gpd.GeoSeries(shapely.get_point(shapely.shortest_line(geoms.values, p), 0), crs=CRS)
    straight = t.distance(p)
    keep = ((straight <= WALK) | (straight.rank(method="first") <= 49)).values
    d = pd.Series(np.inf, index=t.index)
    d[keep], snap = matrix(p, t[keep])
    return d, straight, snap

def outline_points(layer):
    g = layer.geometry
    g = g.where(g.geom_type.str.contains("Line"), g.boundary).segmentize(100)
    return gpd.GeoSeries(gpd.points_from_xy(*g.get_coordinates().values.T), crs=CRS)

beach_pts, coast_pts = outline_points(beaches), outline_points(coastline)

## Compute ORS metrics
Same definitions as notebook 04. The isochrone (1200 m walking distance) gives a second, independent count of features within 15 minutes.

In [4]:
ors, iso, snaps = {}, {}, {}
for fid, b in sample.iterrows():
    p = b.geometry.centroid
    am, am_s, snaps[fid] = ors_layer(p, amenities.geometry)
    gr, _, _ = ors_layer(p, green.geometry)
    st, _, _ = ors_layer(p, stops.geometry)
    hb, _, _ = ors_layer(p, hubs.geometry)
    be, _, _ = ors_layer(p, beach_pts)
    co, _, _ = ors_layer(p, coast_pts)
    ors[fid] = {
        "nd_amenity5_m": am.nsmallest(5).mean(), "nd_green5_m": gr.nsmallest(5).mean(),
        "nd_hub_m": hb.min(), "nd_bus_stop_m": st.min(), "nd_beach_m": be.min(), "nd_coast_m": co.min(),
        "detour_amenity": max(am.min() / max(am_s[am.idxmin()], 50), 1),
        "parks_15min": (gr[is_park] <= WALK).sum(), "amenities_15min": (am <= WALK).sum(),
        "bus_stops_15min": (st <= WALK).sum()}

    ll = gpd.GeoSeries([p], crs=CRS).to_crs(4326)
    r = requests.post(f"{ORS}/isochrones/foot-walking", headers=HEAD, json={
        "locations": [[ll.x[0], ll.y[0]]], "range": [WALK], "range_type": "distance"})
    r.raise_for_status()
    poly = gpd.GeoDataFrame.from_features(r.json(), crs=4326).to_crs(CRS).union_all()
    iso[fid] = {"parks_15min": parks.intersects(poly).sum(), "amenities_15min": amenities.intersects(poly).sum(),
                "bus_stops_15min": stops.intersects(poly).sum()}
    time.sleep(3)

ors, iso = pd.DataFrame(ors).T, pd.DataFrame(iso).T
print("ORS snapped distance (m) - building centroid to ORS road network:", pd.Series(snaps).round(1).to_dict())

ORS snapped distance (m) - building centroid to ORS road network: {5717: 43.1, 34186: 29.7, 42918: 13.9, 31480: 16.4, 75274: 23.0}


## Comparison per building
`notebook` = value in `output/buildings_network.gpkg` (includes the building-to-street snap leg);
`ors` = ORS matrix (excludes the snap leg); `ors_iso` = count inside the ORS isochrone.

In [5]:
cols = list(ors.columns)
cmp = pd.concat({"notebook": sample[cols].stack(), "ors": ors.stack(), "ors_iso": iso.stack()}, axis=1).astype(float)
cmp.index.names = ["fid", "variable"]
cmp["diff"] = cmp["ors"] - cmp["notebook"]
cmp["diff_%"] = 100 * cmp["diff"] / cmp["notebook"].replace(0, np.nan)
pd.set_option("display.max_rows", 100)
cmp.round(2)

notebook      ors  ors_iso     diff  diff_%
fid   variable                                                    
5717  nd_amenity5_m     3025.96  3172.91      NaN   146.95    4.86
      nd_green5_m        324.42   365.31      NaN    40.89   12.60
      nd_hub_m          4837.92  5836.37      NaN   998.45   20.64
      nd_bus_stop_m      262.95   289.97      NaN    27.02   10.28
      nd_beach_m         361.11   318.09      NaN   -43.02  -11.91
      nd_coast_m        2273.06   335.91      NaN -1937.15  -85.22
      detour_amenity       1.19     1.23      NaN     0.04    3.68
      parks_15min          0.00     0.00      0.0     0.00     NaN
      amenities_15min      0.00     0.00      0.0     0.00     NaN
      bus_stops_15min      2.00     2.00      2.0     0.00    0.00
34186 nd_amenity5_m      769.80   882.37      NaN   112.57   14.62
      nd_green5_m        577.41   597.35      NaN    19.94    3.45
      nd_hub_m          1764.21  1967.10      NaN   202.89   11.50
      nd_bus_stop_m      196.37   213.07      NaN    16.70    8.50
      nd_beach_m        4038.19  4180.68      NaN   142.49    3.53
      nd_coast_m        4118.11  4195.52      NaN    77.41    1.88
      detour_amenity       1.27     1.25      NaN    -0.02   -1.51
      parks_15min          1.00     3.00      3.0     2.00  200.00
      amenities_15min      9.00     7.00     10.0    -2.00  -22.22
      bus_stops_15min     23.00    22.00     23.0    -1.00   -4.35
42918 nd_amenity5_m      182.22   167.47      NaN   -14.75   -8.09
      nd_green5_m        117.87   144.78      NaN    26.91   22.83
      nd_hub_m          1486.65  1505.44      NaN    18.79    1.26
      nd_bus_stop_m      310.40   368.86      NaN    58.46   18.83
      nd_beach_m        1962.43  2054.68      NaN    92.25    4.70
      nd_coast_m        2042.34  2054.68      NaN    12.34    0.60
      detour_amenity       1.65     1.59      NaN    -0.06   -3.92
      parks_15min          9.00     8.00     11.0    -1.00  -11.11
      amenities_15min     24.00    26.00     28.0     2.00    8.33
      bus_stops_15min     26.00    24.00     30.0    -2.00   -7.69
31480 nd_amenity5_m     1578.41  1714.59      NaN   136.18    8.63
      nd_green5_m        529.35   535.98      NaN     6.63    1.25
      nd_hub_m          1997.55  2725.38      NaN   727.83   36.44
      nd_bus_stop_m      446.22   443.54      NaN    -2.68   -0.60
      nd_beach_m        5115.60  5147.14      NaN    31.54    0.62
      nd_coast_m        5195.52  5161.98      NaN   -33.54   -0.65
      detour_amenity       1.28     1.51      NaN     0.23   18.33
      parks_15min          3.00     3.00      3.0     0.00    0.00
      amenities_15min      1.00     0.00      1.0    -1.00 -100.00
      bus_stops_15min      5.00     1.00      6.0    -4.00  -80.00
75274 nd_amenity5_m      592.43   653.39      NaN    60.96   10.29
      nd_green5_m        333.59   322.25      NaN   -11.34   -3.40
      nd_hub_m           452.18   451.54      NaN    -0.64   -0.14
      nd_bus_stop_m      169.22   188.83      NaN    19.61   11.59
      nd_beach_m        1121.63  1143.98      NaN    22.35    1.99
      nd_coast_m        1191.03  1146.76      NaN   -44.27   -3.72
      detour_amenity       1.15     1.09      NaN    -0.06   -5.49
      parks_15min          9.00     8.00      9.0    -1.00  -11.11
      amenities_15min      8.00     6.00      8.0    -2.00  -25.00
      bus_stops_15min     24.00    24.00     30.0     0.00    0.00

## Summary per variable

In [6]:
g = cmp.groupby("variable", sort=False)
pd.DataFrame({"mean_notebook": g["notebook"].mean(), "mean_ors": g["ors"].mean(),
              "mean_abs_diff": g["diff"].apply(lambda d: d.abs().mean()),
              "mean_abs_diff_%": g["diff_%"].apply(lambda d: d.abs().mean()),
              "ors_higher (of 5)": g["diff"].apply(lambda d: (d > 0).sum()),
              "mean_abs_diff_iso": (cmp["ors_iso"] - cmp["notebook"]).abs().groupby("variable", sort=False).mean()}).round(2)

,mean_notebook,mean_ors,mean_abs_diff,mean_abs_diff_%,ors_higher (of 5),mean_abs_diff_iso
variable,,,,,,
nd_amenity5_m,1229.76,1318.15,94.28,9.30,4,NaN
nd_green5_m,376.53,393.13,21.14,8.71,4,NaN
nd_hub_m,2107.70,2497.17,389.72,14.00,4,NaN
nd_bus_stop_m,277.03,300.85,24.89,9.96,4,NaN
nd_beach_m,2519.79,2568.91,66.33,4.55,4,NaN
nd_coast_m,2964.01,2578.97,420.94,18.41,2,NaN
detour_amenity,1.31,1.33,0.09,6.58,2,NaN
parks_15min,4.40,4.40,0.80,55.56,1,0.8
amenities_15min,8.40,7.80,1.40,38.89,1,1.0


## Known sources of difference
- **Snap leg**: notebook 04 adds the straight-line distance from the building centroid to its nearest street node; ORS starts at the
  snapped point on the nearest road segment and does not add that leg (see the snapped distances printed above).
- **Network data**: ORS uses its own OSM extract and `foot-walking` profile (which may exclude some ways); notebook 04 uses the
  OSMnx `all` network downloaded in notebook 02, largest connected component only.
- **Polygon and line targets**: notebook 04 links every street node within 100 m of a feature; here each feature is reached at its
  nearest point (beaches/coast: densified outline vertices), so polygon/line distances are approximations.
- **Isochrone counts** depend on ORS's polygon generalisation, so they can differ from the matrix counts near the 1200 m limit.